In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from collections import deque, defaultdict
from torch.distributions import Categorical
from torch.utils.tensorboard import SummaryWriter
import random
import time
import math
from obelix import OBELIX
import pickle

In [ ]:
ACTIONS = ("L45", "L22", "FW", "R22", "R45")

In [ ]:
class PERBuffer:
    def __init__(self, capacity, alpha=0.6, beta=0.4, beta_max=0.9995):
        self.capacity = capacity

        self.buffer = np.empty(capacity, dtype=object)
        self.priorities = np.zeros(capacity, dtype=np.float32)

        self.pos = 0
        self.size = 0

        self.alpha = alpha
        self.beta = beta
        self.beta_max = beta_max
        self.beta_increment = (self.beta_max - self.beta) / 100_000

        self.epsilon = 1e-5
        self.max_priority = 1.0

    def store(self, experience):
        self.buffer[self.pos] = experience
        self.priorities[self.pos] = self.max_priority

        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        current_priorities = self.priorities[:self.size]
        probs = current_priorities / current_priorities.sum()
        indices = np.random.choice(self.size, batch_size, p=probs, replace=True)

        batch = [self.buffer[idx] for idx in indices]
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))

        weights = (self.size * probs[indices]) ** (-self.beta)
        weights = np.array(weights, dtype=np.float32)
        weights /= weights.max()

        self.beta = min(self.beta_max, self.beta + self.beta_increment)

        return states, actions, rewards, next_states, dones, indices, weights

    def update(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors):
            priority = (abs(td_error) + self.epsilon) ** self.alpha

            stored_reward = self.buffer[idx][2]

            if stored_reward >= 80.0:
                priority = max(priority, self.max_priority)

            self.priorities[idx] = priority
            self.max_priority = max(self.max_priority, priority)

    def length(self):
        return self.size

In [ ]:
class DiscreteActor(nn.Module):
    def __init__(self, base_state_dim=18, num_frames=4, action_dim=5):
        super(DiscreteActor, self).__init__()
        self.input_dim = base_state_dim * num_frames

        self.net = nn.Sequential(
            nn.Linear(self.input_dim, 256),
            nn.LeakyReLU(0.01),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.01),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.01),
            nn.Linear(64, action_dim)
        )

    def forward(self, state):
        # state shape: [Batch, 72]
        logits = self.net(state)
        return logits


class TwinCritic(nn.Module):
    def __init__(self, base_state_dim=18, num_frames=4, action_dim=5):
        super(TwinCritic, self).__init__()

        self.input_dim = base_state_dim * num_frames

        def build_critic_path():
            return nn.Sequential(
                nn.Linear(self.input_dim, 256),
                nn.LeakyReLU(0.01),
                nn.Linear(256, 128),
                nn.LeakyReLU(0.01),
                nn.Linear(128, 64),
                nn.LeakyReLU(0.01),
                nn.Linear(64, action_dim)
            )

        self.q1_net = build_critic_path()
        self.q2_net = build_critic_path()

    def forward(self, state):
        # state shape: [Batch, 72]
        q1 = self.q1_net(state)
        q2 = self.q2_net(state)

        return q1, q2

In [ ]:
class RecurrentPPO(nn.Module):
    def __init__(self, action_dim, hidden_size=64):
        super(RecurrentPPO, self).__init__()
        self.hidden_size = hidden_size
        self.action_dim = action_dim

        self.input_layer = nn.Linear(action_dim, hidden_size)

        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

        self.actor = nn.Linear(hidden_size, action_dim)
        self.critic = nn.Linear(hidden_size, 1)

    def forward(self, prev_action, hidden):
        prev_action_onehot = F.one_hot(prev_action, num_classes=self.action_dim).float()
        x = F.relu(self.input_layer(prev_action_onehot))
        x = x.unsqueeze(1)
        x, next_hidden = self.gru(x, hidden)
        x = x.squeeze(1)
        action_logits = self.actor(x)

        return action_logits, None, next_hidden

In [ ]:
class DiscreteSACAgent:
    def __init__(self, env, state_dim=18, action_dim=5, actor_lr=1e-4, critic_lr = 3e-4, gamma=0.995, tau=0.01, alpha=0.2,
                 bufferSize=200_000, alpha_buf=0.6, beta=0.2, beta_rate=0.9, beta_max=0.995, rnd_scale=0.5, num_frames=4,
                 unwedger_path = "best_unwedge_gru.pth", detection_range = 60.0, fov_deg = 45.0, max_penalty = 15.0):
        self.env = env
        self.gamma = gamma
        self.tau = tau
        self.alpha = alpha
        self.action_dim = action_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.actor = DiscreteActor(base_state_dim = state_dim, num_frames=num_frames, action_dim=action_dim).to(self.device)
        self.critic = TwinCritic(base_state_dim=state_dim, num_frames=num_frames, action_dim=action_dim).to(self.device)
        self.critic_target = TwinCritic(base_state_dim=state_dim, num_frames=num_frames, action_dim=action_dim).to(self.device)

        self.critic_target.load_state_dict(self.critic.state_dict())

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr, weight_decay=1e-4)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.unwedger = RecurrentPPO(action_dim=action_dim).to(self.device)
        self.unwedger.load_state_dict(torch.load(unwedger_path, map_location=self.device))
        self.unwedger.eval()

        self.replay_buffer = PERBuffer(capacity=bufferSize, alpha=alpha_buf, beta=beta, beta_max=beta_max)

    def update_odometry(self, action, stuck_bit):
        self.internal_theta = (self.internal_theta + self.theta_deltas[action]) % (2 * math.pi)
        if action == 2 and stuck_bit == 0:
            self.internal_x += math.cos(self.internal_theta)
            self.internal_y += math.sin(self.internal_theta)

    def select_action(self, state, explore=True):
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.actor(state_tensor)
        probs = F.softmax(logits, dim=-1)
        if explore:
            dist = Categorical(probs.squeeze())
            action = dist.sample()
        else:
            action = torch.argmax(probs.squeeze(), dim=-1)

        return action.item()

    def _grad_update(self, states, actions, target_q, weights):
        # states shape: (Batch, State_Dim)
        # actions shape: (Batch, 1)
        # target_q shape: (Batch, 1)
        current_q1, current_q2 = self.critic(states)

        current_q1 = current_q1.gather(-1, actions)
        current_q2 = current_q2.gather(-1, actions)

        critic_loss_1 = F.mse_loss(current_q1, target_q, reduction='none')
        critic_loss_2 = F.mse_loss(current_q2, target_q, reduction='none')

        critic_loss = (weights * (critic_loss_1 + critic_loss_2) / 2.0).mean()

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=1.0)
        self.critic_optimizer.step()

        logits = self.actor(states)
        probs, log_probs = F.softmax(logits, dim=-1), F.log_softmax(logits, dim=-1)

        with torch.no_grad():
            q1, q2 = self.critic(states)
            min_q = torch.min(q1, q2)

        actor_loss = (probs * (self.alpha * log_probs - min_q)).sum(dim=-1).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=1.0)
        self.actor_optimizer.step()

        return critic_loss.item(), actor_loss.item()

    def update(self, batch_size=128, utd_ratio=2):
        c_losses, a_losses, td_errors, q_values_list = [], [], [], []

        for _ in range(utd_ratio):
            states, actions, rewards, next_states, dones, sample_indices, weights = self.replay_buffer.sample(batch_size)

            states = torch.FloatTensor(states).to(self.device)
            actions = torch.LongTensor(actions).unsqueeze(-1).to(self.device)
            rewards = torch.FloatTensor(rewards).unsqueeze(-1).to(self.device) * 0.1
            next_states = torch.FloatTensor(next_states).to(self.device)
            dones = torch.FloatTensor(dones).unsqueeze(-1).to(self.device)
            weights = torch.FloatTensor(weights).unsqueeze(-1).to(self.device)

            with torch.no_grad():
                logits = self.actor(next_states)
                next_probs, next_log_probs = F.softmax(logits, dim=-1), F.log_softmax(logits, dim=-1)
                next_q1, next_q2 = self.critic_target(next_states)
                min_next_q = torch.min(next_q1, next_q2)

                next_v = (next_probs * (min_next_q - self.alpha * next_log_probs)).sum(dim=-1, keepdim=True)
                target_q = rewards + (1 - dones) * self.gamma * next_v

            c_loss, a_loss = self._grad_update(states, actions, target_q, weights)

            with torch.no_grad():
                final_q1, final_q2 = self.critic(states)
                mean_q_vals = final_q1.mean(dim=0).cpu().numpy()
                final_q1 = final_q1.gather(-1, actions)
                final_q2 = final_q2.gather(-1, actions)

                td_errs = (torch.abs(target_q - final_q1) + torch.abs(target_q - final_q2)) / 2.0

            td_errors_numpy = td_errs.squeeze().detach().cpu().numpy()
            self.replay_buffer.update(sample_indices, td_errors_numpy)

            for target_param, param in zip(self.critic_target.parameters(), self.critic.parameters()):
                target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)

            c_losses.append(c_loss)
            a_losses.append(a_loss)
            td_errors.append(td_errors_numpy.mean())
            q_values_list.append(mean_q_vals)

        return np.mean(c_losses), np.mean(a_losses), np.mean(td_errors), np.mean(q_values_list, axis=0)

    def trainAgent(self, max_episodes=200, batch_size=128, utd_ratio=2, MIN_BUF_LEN=1000, alpha_decay=0.98,
                   start_alpha=0.9, min_alpha=0.1, update_freq=4):

        writer = SummaryWriter(log_dir="runs/FSAC_PER_Experiment")
        action_names = ['L45', 'L22', 'FW', 'R22', 'R45']
        self.alpha = start_alpha
        total_steps = 0

        for episode in range(1, max_episodes + 1):
            state = self.env.reset()
            ep_reward, ep_steps = 0, 0
            ep_critic_losses, ep_actor_losses, ep_td_errors, ep_actions, ep_q_vals, rnd_rewards = [], [], [], [], [], []
            done = False
            self.internal_x, self.internal_y, self.internal_theta = 0.0, 0.0, 0.0

            consecutive_spin_steps = 0
            consecutive_stuck_steps = 0
            is_attached = 0.0
            stuck_times = 0
            success = 0
            is_stuck = False

            if self.replay_buffer.length() >= MIN_BUF_LEN:
                self.alpha = max(min_alpha, self.alpha * alpha_decay)

            while not done:

                current_frame = state[-18:]
                is_stuck = (current_frame[17] == 1)

                if is_stuck:
                    # THE ESCAPE POD (GRU TAKEOVER)
                    hidden = torch.zeros(1, 1, self.unwedger.hidden_size).to(self.device)
                    prev_action = torch.tensor([0]).to(self.device)
                    while current_frame[17] == 1 and not done:
                        stuck_times += 1
                        with torch.no_grad():
                            logits, _, hidden = self.unwedger(prev_action, hidden)
                            dist = Categorical(F.softmax(logits, dim=-1).squeeze())
                            action = dist.sample().item()

                        next_state, env_reward, done = self.env.step(ACTIONS[action], render=False)

                        current_frame = next_state[-18:]
                        self.update_odometry(action, current_frame[17])

                        prev_action = torch.tensor([action]).to(self.device)
                        state = next_state
                        total_steps += 1
                        ep_steps += 1
                        ep_reward += env_reward
                        ep_actions.append(action)

                    continue

                action = self.select_action(state, explore=True)
                next_state, env_reward, done = self.env.step(ACTIONS[action], render=False)

                if action == 2:
                    consecutive_spin_steps = 0
                else:
                    consecutive_spin_steps += 1

                base_reward = -10.0

                if env_reward <= -150.0:
                    base_reward = -500.0
                elif env_reward >= 1000.0:
                    base_reward = 1000.0
                    success = 1.0
                elif env_reward >= 80:
                    base_reward = 100.0
                    is_attached = 1.0

                phase_reward = 0.0
                infra_active = current_frame[16] == 1
                right_sonars = np.max(current_frame[0:4]) == 1
                front_sonars = np.max(current_frame[4:12]) == 1
                left_sonars  = np.max(current_frame[12:16]) == 1

                if is_attached:
                    phase_reward = 50.0 if action == 2 else -20.0
                elif infra_active or front_sonars:
                    phase_reward = 20.0 if action == 2 else -20.0
                elif left_sonars:
                    phase_reward = 20.0 if action in [0, 1] else -20.0
                elif right_sonars:
                    phase_reward = 20.0 if action in [3, 4] else -20.0
                else:
                    phase_reward = 5.0 if action == 2 else -10.0 * consecutive_spin_steps

                shaped_reward = base_reward + phase_reward

                self.replay_buffer.store((state, action, shaped_reward, next_state, done))

                ep_actions.append(action)
                ep_reward += env_reward
                ep_steps += 1
                total_steps += 1
                state = next_state

                # --- TRAINING ---
                if self.replay_buffer.length() >= MIN_BUF_LEN and total_steps % update_freq == 0:
                    c_loss, a_loss, td_err, q_vals = self.update(batch_size=batch_size, utd_ratio=utd_ratio)
                    ep_critic_losses.append(c_loss)
                    ep_actor_losses.append(a_loss)
                    ep_td_errors.append(td_err)
                    ep_q_vals.append(q_vals)

            writer.add_scalar('Environment/Total_Reward', ep_reward, episode)
            writer.add_scalar('Environment/Episode_Steps', ep_steps, episode)
            writer.add_scalar('Environment/Alpha', self.alpha, episode)
            writer.add_scalar('Environment/Success', success, episode)
            writer.add_scalar('Environment/Attached', is_attached, episode)
            writer.add_scalar('Environment/Stuck Times', stuck_times, episode)
            action_counts = np.bincount(ep_actions, minlength=5)
            for a_idx, count in enumerate(action_counts):
                percentage = (count / ep_steps) * 100
                writer.add_scalar(f'Actions_Percentage/{action_names[a_idx]}', percentage, episode)

            if len(ep_critic_losses) > 0:
                writer.add_scalar('Network_Loss/Critic', np.mean(ep_critic_losses), episode)
                writer.add_scalar('Network_Loss/Actor', np.mean(ep_actor_losses), episode)
                writer.add_scalar('Network_Loss/Average_PER_TD_Error', np.mean(ep_td_errors), episode)
                avg_q_vals = np.mean(ep_q_vals, axis=0)
                for i in range(5):
                      writer.add_scalar(f'Q_Values/{action_names[i]}', avg_q_vals[i], episode)

    def load_model(self, filepath):
        checkpoint = torch.load(filepath)
        self.actor.load_state_dict(checkpoint['actor_state_dict'])
        self.critic.load_state_dict(checkpoint['critic_state_dict'])
        self.critic_target.load_state_dict(checkpoint['critic_target_state_dict'])

        # Restore the optimizers (crucial if you plan to resume training)
        # self.actor_optimizer.load_state_dict(checkpoint['actor_optimizer'])
        # self.critic_optimizer.load_state_dict(checkpoint['critic_optimizer'])

    def evaluateAgent(self, episodes=5):
        action_map, reward_map = defaultdict(int), defaultdict(int)
        for episode in range(episodes):
            state = self.env.reset()
            done = False
            ep_reward, ep_steps = 0, 0
            action_map.clear()
            reward_map.clear()

            while not done:
                action = self.select_action(state, explore=False)

                next_state, reward, done = self.env.step(ACTIONS[action], render=False)
                action_map[action] += 1
                reward_map[reward] += 1
                ep_steps += 1
                ep_reward += reward
                state = next_state

            print(f"Action Map: {dict(action_map)} | Reward Map: {dict(reward_map)}")
            print(f"Episode: {episode} | Reward: {ep_reward:.1f} | Steps: {ep_steps}")

In [ ]:
class FrameStackWrapper:
    def __init__(self, env, num_frames=4, state_dim=18):
        self.env = env
        self.num_frames = num_frames
        self.original_state_dim = state_dim
        self.state_dim = self.num_frames * self.original_state_dim
        self.frames = deque(maxlen=num_frames)

    def reset(self):
        obs = self.env.reset()
        for _ in range(self.num_frames):
            self.frames.append(obs)
        return self._get_stacked_obs()

    def step(self, action, render=False):
        obs, reward, done = self.env.step(action, render=render)
        self.frames.append(obs)
        return self._get_stacked_obs(), reward, done

    def _get_stacked_obs(self):
        return np.concatenate(self.frames, axis=0)

In [ ]:
env = OBELIX(scaling_factor = 5,
             arena_size = 500,
             max_steps = 2000,
             wall_obstacles = False,
             seed = 0,
             difficulty = 0)
env = FrameStackWrapper(env = env)
agent = DiscreteSACAgent(env = env, state_dim = 18, action_dim = 5, gamma = 0.999, tau = 0.01, alpha = 0.05,
                         bufferSize = 500_000, alpha_buf = 1.0, beta = 0.2, beta_max = 0.9999, actor_lr = 1e-4, critic_lr =3e-4,
                         unwedger_path="/content/best_unwedge_gru_latest.pth")

# agent.load_model('/content/FSAC_with_gru_500_no_walls.pth')

In [ ]:
# # Load the TensorBoard notebook extension
%reload_ext tensorboard

# # Start TensorBoard, pointing it to the 'runs' folder we are about to create
%tensorboard --logdir runs

In [ ]:
agent.trainAgent(max_episodes = 1000, batch_size = 256, utd_ratio = 1, MIN_BUF_LEN = 10_000,
                 alpha_decay = 0.995, start_alpha = 1.0, min_alpha = 0.2, update_freq = 5)

In [ ]:
checkpoint = {
            'actor_state_dict': agent.actor.state_dict(),
            'critic_state_dict': agent.critic.state_dict(),
            'critic_target_state_dict': agent.critic_target.state_dict(),
            # 'actor_optimizer': agent.actor_optimizer.state_dict(),
            # 'critic_optimizer': agent.critic_optimizer.state_dict(),
            'unwedger': agent.unwedger.state_dict(),
        }
torch.save(checkpoint, "FSAC_with_gru_no_walls_with_wd.pth")

In [ ]:
agent.evaluateAgent(episodes = 10)